<a href="https://colab.research.google.com/github/marsanla/C4L/blob/main/SOL_C4L_4_IA%2C_LLMs_y_prompt_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Soluciones — Módulo 4: IA y prompt engineering

> **No hay una única solución correcta** en prompt engineering. Lo que verás abajo son **ejemplos de prompts bien construidos** que aplican los 4 ingredientes (rol, tarea, formato, ejemplos) y los patrones que vimos en el módulo. Comparadlos con los tus y quedaos con las ideas que más te convenzan.


### Ejercicio 1 — Convertir prompt débil en fuerte

Prompt original (débil):

```
Lee este contrato y dime si es bueno.
```

#### Solución propuesta — versión fuerte

```
Eres un abogado mercantil con experiencia en revisión de contratos
B2B en España.

CONTEXTO:
Mi cliente, "EmpresaX S.L.", figura como ARRENDATARIA en el contrato
adjunto. Quiero saber si las condiciones son aceptables para ella.

TAREA:
Analiza el contrato bajo dos criterios:

  1. Equilibrio de obligaciones (¿hay cláusulas claramente desfavorables
     para mi cliente?).
  2. Conformidad con la legislación española vigente, especialmente:
       - LAU si es arrendamiento.
       - Código de Comercio en cláusulas mercantiles.
       - LCGC en cláusulas estandarizadas.

Para cada criterio:
  - Indica el nivel de riesgo (bajo, medio, alto).
  - Identifica las cláusulas concretas (con su numeración).
  - Sugiere la redacción alternativa si procede.

FORMATO:
Devuelve una tabla Markdown con columnas: cláusula, problema, riesgo,
sugerencia. Después de la tabla, añade un párrafo final con la
recomendación general (firmar / negociar / no firmar).

Si te falta información para evaluar algún punto, indica claramente
"No consta en el contrato — pedir aclaración" en lugar de inventar.

CONTRATO:
[texto del contrato]
```

> **Por qué es mejor:** define rol, contexto, criterios concretos, formato
> de salida y mecanismo anti-alucinación ("no consta en el contrato").


### Ejercicio 2 — Salida en JSON

#### Solución propuesta

```
Eres un asistente que extrae información estructurada de contratos.

TAREA:
Lee el contrato adjunto y extrae estos campos:

  - partes:           lista de objetos con {nombre, rol}
  - objeto:           descripción breve (string, máx. 200 caracteres)
  - cuantia:          número (entero o decimal). Si no consta, usa null.
  - fecha_inicio:     "AAAA-MM-DD". Si no consta, usa null.
  - fecha_vencimiento:"AAAA-MM-DD". Si no consta, usa null.
  - riesgos:          lista de strings con cláusulas problemáticas detectadas
                      (vacía si no hay riesgos).

REGLAS DE SALIDA:
  - Devuelve EXCLUSIVAMENTE el JSON, sin texto antes ni después.
  - No incluyas comentarios dentro del JSON.
  - Usa null (no "null", no "N/A") cuando un campo no se pueda obtener.
  - Las fechas DEBEN seguir exactamente el formato AAAA-MM-DD.

EJEMPLO DE FORMATO ESPERADO:
{
  "partes": [
    {"nombre": "Empresa A S.L.", "rol": "arrendadora"},
    {"nombre": "Empresa B S.A.", "rol": "arrendataria"}
  ],
  "objeto": "Arrendamiento de local de negocio en Madrid",
  "cuantia": 24000,
  "fecha_inicio": "2024-01-01",
  "fecha_vencimiento": "2026-12-31",
  "riesgos": [
    "Cláusula 12: penalización desproporcionada por terminación anticipada"
  ]
}

CONTRATO:
[texto del contrato]
```

> **Por qué es mejor:** especifica nombres de campos, tipos, valores nulos,
> formato de fechas, e incluye un ejemplo de salida — todo crítico cuando la
> respuesta va a ser parseada por un script Python con `json.loads()`.


### Ejercicio 3 — Anti-alucinación

#### Solución propuesta

```
Eres un asistente jurídico de soporte. Tu cometido es ayudarme a localizar
jurisprudencia española sobre cláusulas suelo en hipotecas.

REGLAS ESTRICTAS:
  1. NO inventes referencias. Si no estás seguro de que una sentencia existe
     y es exactamente como la describes, NO la incluyas.
  2. Para cada referencia que aportes, indica:
       - Tribunal y sala.
       - Fecha (al menos año).
       - Número de procedimiento si lo sabes con seguridad.
       - Tu nivel de confianza: ALTO / MEDIO / BAJO.
  3. Si tu nivel de confianza es BAJO o MEDIO, marca la referencia con
     [VERIFICAR ANTES DE USAR] al final.
  4. Si no tienes suficiente información, responde literalmente:
     "No tengo información suficiente para responder con seguridad."

ALCANCE:
  - Solo jurisprudencia del Tribunal Supremo y del TJUE.
  - Solo sentencias relevantes para la cláusula suelo en hipotecas.
  - Excluye comentarios doctrinales — me interesan SOLO sentencias.

FORMATO:
Tabla Markdown con columnas: tribunal, sala, fecha, asunto, tema clave,
nivel de confianza, observaciones.

PREGUNTA: <tu pregunta concreta>
```

> **Por qué es mejor:** la combinación "no inventes / di no sé / nivel de confianza"
> reduce drásticamente las alucinaciones. Aun así, RECUERDA: este tipo de salida
> SIEMPRE hay que verificarla contra la fuente original.


### Ejercicio 4 — Reflexión

> Las respuestas a este ejercicio son personales — cada profesional tiene tareas distintas. A modo de ejemplo, un patrón típico de respuestas:

#### 3 tareas que la IA puede asumir total o parcialmente

1. **Resumir documentos largos** (sentencias, dictámenes, contratos extensos): la IA produce un primer resumen que el abogado revisa y matiza. Ahorra el 80% del tiempo de lectura inicial.
2. **Reformular un texto** en otro tono o longitud: por ejemplo, convertir un dictamen técnico en un email para el cliente que lo entienda. La IA es excelente en esto.
3. **Extraer datos estructurados de PDFs**: nombres, cuantías, fechas, partes. Bien instruida, la IA reduce horas de copia manual a minutos.

#### 2 tareas donde la IA NO debería intervenir

1. **Decisión final de un caso o estrategia procesal**: requiere juicio profesional, conocimiento del cliente, intuición jurídica y responsabilidad legal. Es trabajo del letrado.
2. **Citas de jurisprudencia que no se vayan a verificar**: la IA inventa referencias con regularidad. Si no hay tiempo o herramienta para verificar, no la uses.

#### Caso donde Python + IA superan a cualquiera por separado

> **Ejemplo:** *"Auditar 5.000 contratos buscando cláusulas que vulneren la nueva normativa europea de servicios digitales y generar un informe ejecutivo."*
>
> - **Solo la IA**: tendrías que pegar contrato a contrato en ChatGPT — inviable a esa escala.
> - **Solo Python**: podría buscar palabras clave, pero no detecta matices ni contexto.
> - **Python + IA**: Python lee y prepara los textos, los envía a la API del modelo con un prompt afinado, agrega los resultados en un DataFrame y genera el informe ejecutivo. **Todo en una tarde.**


## Cierra del módulo

Estos ejemplos cubren lo básico — en el **curso de pago** entraremos en:

- Construcción de prompts maestros reutilizables.
- Few-shot prompting con ejemplos curados.
- Chain-of-thought y razonamiento estructurado.
- Function calling para que el modelo llame a tus funciones Python.
- Construcción de pipelines completos (Python + IA + datos).
- RAG: conectar el modelo a tu base interna de conocimiento.
- Compliance y privacidad en proyectos reales.
